# Chapter 22 — Automated Configuration Generation
## From Design Intent to Device-Ready CLI — Without Copy-Paste

You have a beautiful branch office design document: OSPF areas, VLAN structure,
QoS policies, AAA config. The first engineer implements it one way.
The second engineer implements it slightly differently.
By the fifteenth site, the "standard" has **fifteen variations**.

Configuration drift starts on **day one**, before a single change ticket is raised.

The root cause: the translation from *design intent* to *device configuration*
is always manual, always interpreted, always subject to individual variation.

Two approaches exist to fix this:

| Approach | Analogy | Strengths | Weaknesses |
|----------|---------|-----------|------------|
| **Template-Based (Jinja2)** | Fill-in form | Deterministic, auditable | Limited to what the form covers |
| **LLM Synthesis** | Expert consultant | Flexible, multi-vendor, novel requirements | Probabilistic — may hallucinate commands |

The right answer: **use both**. Templates for the predictable 80%, LLM for the other 20%.

> **No GPU needed.** Uses Anthropic API + Python's built-in `string.Template`.

### Install
```
pip install anthropic
```


## Setup — API Client and Knowledge Base

In [ ]:
import json, re, string
from anthropic import Anthropic

# ── API Key Setup (Colab + local fallback) ────────────────────────────────────
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    api_key = getpass.getpass("Enter your Anthropic API key: ")

client = Anthropic(api_key=api_key)

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

def ask(prompt, model=HAIKU, system="You are a senior network automation engineer."):
    r = client.messages.create(
        model=model, max_tokens=1200, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

# ── Simulated vendor command reference (what RAG would retrieve) ───────────────
# In production: pulled from Cisco DevNet docs, Juniper TechLibrary, etc.
VENDOR_DOCS = {
    "Cisco IOS": {
        "ospf_auth_md5": "ip ospf message-digest-key {key_id} md5 {key}\nip ospf authentication message-digest",
        "ospf_auth_area": "area {area} authentication message-digest",
        "bgp_neighbor":   "neighbor {ip} remote-as {asn}\nneighbor {ip} update-source {intf}",
        "vlan":           "vlan {id}\n name {name}",
        "deprecated": ["ip ospf authentication md5",   # wrong — does not exist on IOS
                       "neighbor {ip} advertisement-interval"],  # deprecated
    },
    "Cisco IOS-XE": {
        "ospf_auth_md5": "ip ospf authentication message-digest\nip ospf message-digest-key {key_id} md5 {key}",
        "ospf_auth_area": "area {area} authentication message-digest",
        "bgp_neighbor":   "neighbor {ip} remote-as {asn}\naddress-family ipv4\n  neighbor {ip} activate",
        "vlan":           "vlan {id}\n name {name}",
        "deprecated": [],
    },
    "Juniper JunOS": {
        "ospf_auth_md5": "set protocols ospf area {area} interface {intf} authentication md5 {key_id} key {key}",
        "bgp_neighbor":   "set protocols bgp group EBGP neighbor {ip} peer-as {asn}",
        "vlan":           "set vlans {name} vlan-id {id}",
        "deprecated": [],
    },
}

print("Knowledge base loaded:")
for vendor in VENDOR_DOCS:
    cmds = [k for k in VENDOR_DOCS[vendor] if k != "deprecated"]
    print(f"  {vendor}: {len(cmds)} command patterns")


---
## Demo 1 — Template-Based Generation: Deterministic but Limited

Template-based generation has been the backbone of network automation for years.
Tools like **Ansible**, **Salt**, and **NetBox** all use Jinja2 under the hood.

The key property: **determinism** — same inputs always produce byte-for-byte
identical configuration. Perfect for GitOps, audits, and compliance.

But templates have a hard limit: they can only produce what they were
designed to produce. Novel requirements require modifying the template,
which accumulates complexity over time.

```
Branch office design intent
        │
   [Variable substitution]
        │
  ┌─────┼─────┐
  R1    R2    R3   ← identical structure, different values
```


In [ ]:
# ── Template-Based Configuration Generation ───────────────────────────────────
# Using Python's built-in string.Template (same concept as Jinja2, no extra install)

# ── Templates for different config blocks ─────────────────────────────────────

OSPF_TEMPLATE = string.Template("""\
! OSPF Configuration — generated from template
router ospf $process_id
 router-id $router_id
 area $area authentication message-digest
 passive-interface default
 no passive-interface $wan_interface
!
interface $wan_interface
 ip ospf $process_id area $area
 ip ospf message-digest-key $key_id md5 $key
 ip ospf network point-to-point
!""")

BGP_TEMPLATE = string.Template("""\
! BGP Configuration — generated from template
router bgp $local_asn
 bgp router-id $router_id
 neighbor $peer_ip remote-as $peer_asn
 neighbor $peer_ip description $peer_name
 !
 address-family ipv4
  neighbor $peer_ip activate
  neighbor $peer_ip soft-reconfiguration inbound
 exit-address-family
!""")

VLAN_TEMPLATE = string.Template("""\
! VLAN Configuration — generated from template
vlan $vlan_id
 name $vlan_name
!
interface Vlan$vlan_id
 description $vlan_name
 ip address $ip_address $subnet_mask
 no shutdown
!""")

# ── Branch office data (what you'd pull from NetBox/IPAM) ─────────────────────
BRANCH_SITES = [
    {"site": "BRANCH-01", "router_id": "10.0.1.1", "wan_intf": "GigabitEthernet0/0",
     "ospf_pid": 1, "area": 10, "key_id": 1, "key": "0sp3Auth!",
     "local_asn": 65001, "peer_ip": "203.0.113.1", "peer_asn": 65000,
     "peer_name": "ISP-PRIMARY",
     "vlans": [{"id": 100, "name": "DATA", "ip": "192.168.1.1", "mask": "255.255.255.0"},
               {"id": 200, "name": "VOICE", "ip": "192.168.2.1", "mask": "255.255.255.0"}]},
    {"site": "BRANCH-02", "router_id": "10.0.2.1", "wan_intf": "GigabitEthernet0/0",
     "ospf_pid": 1, "area": 10, "key_id": 1, "key": "0sp3Auth!",
     "local_asn": 65002, "peer_ip": "203.0.113.5", "peer_asn": 65000,
     "peer_name": "ISP-PRIMARY",
     "vlans": [{"id": 100, "name": "DATA", "ip": "192.168.3.1", "mask": "255.255.255.0"},
               {"id": 200, "name": "VOICE", "ip": "192.168.4.1", "mask": "255.255.255.0"}]},
]


def generate_from_template(site: dict) -> str:
    """Generate full branch config from templates — deterministic and consistent."""
    parts = [f"! Site: {site['site']} — auto-generated from template", "!"]

    # OSPF block
    parts.append(OSPF_TEMPLATE.substitute(
        process_id=site["ospf_pid"], router_id=site["router_id"],
        area=site["area"], wan_interface=site["wan_intf"],
        key_id=site["key_id"], key=site["key"]
    ))

    # BGP block
    parts.append(BGP_TEMPLATE.substitute(
        local_asn=site["local_asn"], router_id=site["router_id"],
        peer_ip=site["peer_ip"], peer_asn=site["peer_asn"],
        peer_name=site["peer_name"]
    ))

    # VLAN blocks
    for v in site["vlans"]:
        parts.append(VLAN_TEMPLATE.substitute(
            vlan_id=v["id"], vlan_name=v["name"],
            ip_address=v["ip"], subnet_mask=v["mask"]
        ))

    return "\n".join(parts)


# Generate for all branch sites
for site in BRANCH_SITES:
    config = generate_from_template(site)
    print(f"{'='*60}")
    print(config)
    print()

print("Key insight: Both configs are IDENTICAL in structure — only values differ.")
print("This eliminates configuration drift from day-1 deployment.")


---
## Demo 2 — LLM Configuration Synthesis: Flexible but Needs Validation

Templates fail when requirements are novel — anything outside the template's
scope requires modifying the template itself.

**LLM synthesis** has no such limit. It can handle:
- Novel feature combinations no template was written for
- Multi-vendor output from the same intent description
- Cross-platform syntax differences (IOS vs IOS-XE vs JunOS)

The tradeoff: LLMs are **probabilistic**. They can hallucinate commands that:
- Don't exist on the target platform
- Were deprecated in a newer software version
- Are syntactically plausible but semantically wrong

This is the **knowledge boundary problem** — the LLM doesn't know what it doesn't know.


In [ ]:
# ── LLM Multi-Vendor Configuration Synthesis ─────────────────────────────────

def llm_synthesize(intent: str, platforms: list[str]) -> dict[str, str]:
    """
    Generate platform-specific config from a single natural language intent.
    Returns a config string per platform.
    """
    configs = {}
    for platform in platforms:
        prompt = f"""Generate a complete, production-ready configuration snippet for:

Platform: {platform}
Intent: {intent}

Requirements:
- Use ONLY commands that exist on {platform}
- Include interface-level and global-level commands as needed
- Add brief comments (! lines) to explain each section
- Be precise with syntax — no guessing

Output ONLY the CLI configuration. No explanation text.
"""
        configs[platform] = ask(prompt, model=SONNET,
                                system=f"You are a {platform} configuration expert. Output only CLI config.")
    return configs


def show_multi_vendor(intent: str):
    print(f"Intent: {intent}")
    print("=" * 60)

    platforms = ["Cisco IOS", "Cisco IOS-XE", "Juniper JunOS"]
    configs   = llm_synthesize(intent, platforms)

    for platform, config in configs.items():
        print(f"\n{'─'*50}")
        print(f"Platform: {platform}")
        print('─'*50)
        print(config)

    print("\n✓ Same intent → 3 platform-specific configs")
    print("  Template approach would need 3 separate templates, maintained separately.")
    return configs


multi_configs = show_multi_vendor(
    "Configure BGP with the upstream ISP (peer IP 203.0.113.1, peer-AS 65000) "
    "and apply route-map to prepend our AS twice on the outbound path to de-prefer this link"
)


---
## Demo 3 — Hallucination Detection: Catch Wrong Commands Before They Break Things

The biggest risk with LLM config generation: **hallucinated commands**.

Two failure patterns from the chapter:
1. **Snowballing hallucination** — wrong initial assumption propagates. LLM thinks
   it's generating for IOS-XR but it's actually IOS-XE. Every subsequent command
   is "correct for the wrong platform."
2. **Knowledge boundary problem** — deprecated commands that appeared in old training data
   get generated confidently. The LLM doesn't know the command was removed in IOS 17.5.

The fix: a **Critic agent** that acts like a QA engineer:
- Receives the generated config
- Checks each command against the vendor knowledge base
- Flags suspicious commands and rates confidence
- Blocks deployment if confidence is too low


In [ ]:
# ── Hallucination Detection (Critic Agent) ────────────────────────────────────

def check_for_hallucinations(config: str, platform: str) -> dict:
    """
    Critic agent: validates generated config against vendor documentation.
    Returns a report with flagged lines and an overall confidence score.
    """
    # Get known deprecated commands for this platform
    known_deprecated = VENDOR_DOCS.get(platform, {}).get("deprecated", [])

    prompt = f"""You are a strict network configuration reviewer for {platform}.

Generated configuration to review:
{config}

Known deprecated/incorrect commands to flag:
{known_deprecated}

Review EVERY command and check:
1. Does this command exist on {platform}?
2. Is it syntactically correct for this platform?
3. Is it semantically correct (right parameters, right context)?
4. Is it deprecated or changed in recent versions?

Return ONLY this JSON:
{{
  "platform": "{platform}",
  "flagged_lines": [
    {{"line": "the command", "issue": "what is wrong", "severity": "ERROR|WARNING|INFO"}}
  ],
  "confidence_score": 85,
  "safe_to_deploy": true,
  "summary": "one sentence verdict"
}}
"""
    raw = ask(prompt, model=SONNET,
              system="You are a strict network configuration validator. Be precise and conservative.")

    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return {"error": "Could not parse validation result", "raw": raw[:200]}
    try:
        return json.loads(match.group())
    except Exception as e:
        return {"error": str(e), "raw": raw[:200]}


def validate_all_configs(configs: dict[str, str]):
    print("Validating generated configurations for hallucinations...")
    print("=" * 60)

    all_safe = True
    for platform, config in configs.items():
        report = check_for_hallucinations(config, platform)

        if "error" in report:
            print(f"\n[{platform}] Validation error: {report['error']}")
            continue

        safe   = report.get("safe_to_deploy", False)
        score  = report.get("confidence_score", 0)
        flags  = report.get("flagged_lines", [])
        symbol = "✅" if safe else "🔴"

        print(f"\n{symbol} {platform}")
        print(f"   Confidence: {score}%  |  Safe to deploy: {safe}")
        print(f"   {report.get('summary', '')}")

        if flags:
            print(f"   Flagged issues ({len(flags)}):")
            for f in flags:
                sev = {"ERROR": "✗", "WARNING": "⚠", "INFO": "ℹ"}.get(f.get("severity"), "?")
                print(f"     {sev} [{f.get('severity')}] {f.get('line', '')[:60]}")
                print(f"       → {f.get('issue', '')}")
        else:
            print("   No issues flagged.")

        if not safe:
            all_safe = False

    print(f"\n{'─'*60}")
    print(f"Overall verdict: {'✅ ALL CONFIGS SAFE' if all_safe else '🔴 ISSUES FOUND — Review before deploying'}")
    return all_safe


validate_all_configs(multi_configs)


---
## Demo 4 — RAG-Augmented Config Generation: Ground the LLM in Documentation

The cure for hallucination is **RAG for configuration synthesis**.

Instead of the LLM relying on what it memorized during training (which may be stale),
we:
1. **Retrieve** the relevant vendor documentation for the target platform and feature
2. **Augment** the prompt with that documentation as context
3. **Generate** configuration grounded in what was retrieved — not in memory

```
Without RAG:  "Configure OSPF auth" → LLM recalls from training (may be outdated)
                                        → generates deprecated command confidently

With RAG:     "Configure OSPF auth" → retrieve IOS-XE 17.9 auth commands
              [docs injected]        → LLM uses ONLY retrieved commands
                                        → no deprecated syntax possible
```

This is the same principle as Chapter 14's RAG Fundamentals, applied to
config generation instead of Q&A — the LLM can't hallucinate commands that
aren't in the retrieved context.


In [ ]:
# ── RAG-Augmented Configuration Generation ───────────────────────────────────

def retrieve_docs(platform: str, feature: str) -> str:
    """
    Simulated retrieval from vendor documentation knowledge base.
    In production: vector similarity search against Cisco DevNet / Juniper TechLibrary.
    """
    platform_docs = VENDOR_DOCS.get(platform, {})

    # Map feature keywords to doc entries
    retrieved = []
    feature_lower = feature.lower()

    if "ospf" in feature_lower and "auth" in feature_lower:
        if "ospf_auth_md5" in platform_docs:
            retrieved.append(f"Interface-level MD5 auth:\n  {platform_docs['ospf_auth_md5']}")
        if "ospf_auth_area" in platform_docs:
            retrieved.append(f"Area-level auth:\n  {platform_docs['ospf_auth_area']}")
    if "bgp" in feature_lower:
        if "bgp_neighbor" in platform_docs:
            retrieved.append(f"BGP neighbor config:\n  {platform_docs['bgp_neighbor']}")
    if "vlan" in feature_lower:
        if "vlan" in platform_docs:
            retrieved.append(f"VLAN definition:\n  {platform_docs['vlan']}")

    if not retrieved:
        return f"[No specific docs found for '{feature}' on {platform} — use general knowledge carefully]"

    return "\n".join(retrieved)


def rag_config_generate(intent: str, platform: str, feature: str) -> str:
    """
    RAG-augmented generation: inject retrieved docs before asking for config.
    The model is CONSTRAINED to use only what was retrieved.
    """
    docs = retrieve_docs(platform, feature)

    prompt = f"""Generate configuration for {platform}.

Intent: {intent}

RETRIEVED VENDOR DOCUMENTATION (use ONLY these commands — do not invent others):
{docs}

Generate the complete configuration snippet using ONLY the commands shown above.
Substitute the actual values (IP addresses, keys, etc.) from the intent.
Add ! comment lines to explain each section.
Output only CLI configuration.
"""
    return ask(prompt, model=SONNET,
               system=f"You are a {platform} config generator. Use ONLY the commands provided in the documentation above.")


def compare_rag_vs_plain(intent: str, platform: str, feature: str):
    print(f"Intent: {intent}")
    print(f"Platform: {platform}")
    print("=" * 60)

    # Plain LLM (memory only)
    plain = ask(
        f"Generate {platform} configuration for: {intent}\nOutput only CLI.",
        model=HAIKU,
        system=f"You are a {platform} expert."
    )

    # RAG-augmented
    rag = rag_config_generate(intent, platform, feature)

    print("\n[Plain LLM — from memory only]")
    print(plain)

    print("\n" + "─"*60)
    print("[RAG-Augmented — grounded in retrieved documentation]")
    print("\nRetrieved docs used:")
    docs = retrieve_docs(platform, feature)
    for line in docs.splitlines():
        print(f"  {line}")
    print("\nGenerated config:")
    print(rag)

    print("\n─"*60)
    print("RAG advantage: config is constrained to documented commands only.")
    print("Hallucinated commands cannot appear if they weren't in the retrieved context.")
    return rag


rag_result = compare_rag_vs_plain(
    intent="Enable OSPF MD5 authentication on interface GigabitEthernet0/0, area 0, key-id 1, key 'SecureKey99'",
    platform="Cisco IOS-XE",
    feature="ospf authentication"
)


---
## Demo 5 — Full Config Pipeline: Intent to Validated, Deployment-Ready Config

The complete automated configuration generation system:

```
Engineer types: "Deploy standard branch config at BRANCH-03"
        │
   1. Template generation (deterministic base config)
        │
   2. LLM synthesis (novel or cross-vendor requirements)
        │
   3. RAG grounding (inject vendor docs to prevent hallucination)
        │
   4. Hallucination check (Critic agent validates every command)
        │
   5. Config diff vs existing (what will actually change?)
        │
   6. Deployment-ready output (annotated, with rollback commands)
```

This addresses all five failure modes from the chapter intro:
- **Consistent output**: templates guarantee no drift
- **Novel requirements**: LLM handles what templates can't
- **Hallucination risk**: RAG + Critic block bad commands
- **Audit trail**: auto-generated diff and change annotation


In [ ]:
# ── Full Configuration Generation Pipeline ───────────────────────────────────

def config_diff(old_config: str, new_config: str) -> str:
    """
    Simple line-by-line diff — shows what will change.
    In production: use 'difflib' or push to a config management system.
    """
    old_lines = set(old_config.strip().splitlines())
    new_lines = set(new_config.strip().splitlines())
    added     = [l for l in new_lines if l not in old_lines and l.strip()]
    removed   = [l for l in old_lines if l not in new_lines and l.strip()]
    return {
        "added":   added[:10],    # cap for display
        "removed": removed[:10],
        "net_change": len(added) - len(removed),
    }


def full_config_pipeline(site_intent: str, platform: str,
                          existing_config: str = "") -> dict:
    print(f"CONFIG PIPELINE")
    print(f"Intent: {site_intent}")
    print(f"Platform: {platform}")
    print("=" * 70)

    result = {"intent": site_intent, "platform": platform}

    # ── Stage 1: Template base (predictable 80%) ──────────────────────────────
    print("\n[1/5] Template-based base config...")
    if BRANCH_SITES:
        template_config = generate_from_template(BRANCH_SITES[0])
        result["template_lines"] = len(template_config.splitlines())
        print(f"  ✓ Template generated: {result['template_lines']} lines")
    else:
        template_config = ""
        print("  (No template data — skipping)")

    # ── Stage 2: LLM synthesis for novel parts ────────────────────────────────
    print("\n[2/5] LLM synthesis for novel/custom requirements...")
    llm_config = ask(
        f"Generate {platform} config for: {site_intent}\nOutput only CLI.",
        model=SONNET,
        system=f"You are a {platform} network engineer. Output only CLI configuration."
    )
    result["llm_lines"] = len(llm_config.splitlines())
    print(f"  ✓ LLM synthesized: {result['llm_lines']} lines")

    # ── Stage 3: RAG grounding ────────────────────────────────────────────────
    print("\n[3/5] RAG grounding — injecting vendor documentation...")
    # Detect which features the intent mentions
    feature = "ospf bgp vlan" if any(k in site_intent.lower()
                                      for k in ["ospf","bgp","vlan"]) else "general"
    rag_config = rag_config_generate(site_intent, platform, feature)
    result["rag_grounded"] = True
    print(f"  ✓ RAG-grounded config generated")

    # ── Stage 4: Hallucination check ─────────────────────────────────────────
    print("\n[4/5] Hallucination check (Critic agent)...")
    validation = check_for_hallucinations(rag_config, platform)
    confidence = validation.get("confidence_score", 0)
    safe       = validation.get("safe_to_deploy", False)
    flags      = validation.get("flagged_lines", [])
    result["validation"] = {"confidence": confidence, "safe": safe, "flags": len(flags)}
    print(f"  Confidence: {confidence}%  |  Safe: {safe}  |  Issues: {len(flags)}")
    for f in flags:
        sev = f.get("severity", "INFO")
        print(f"  {'✗' if sev=='ERROR' else '⚠'} {f.get('issue','')}")

    if not safe:
        print("  🔴 Config BLOCKED — hallucinations detected. Pipeline aborted.")
        result["status"] = "BLOCKED"
        return result

    # ── Stage 5: Diff vs existing ─────────────────────────────────────────────
    print("\n[5/5] Computing config diff...")
    diff = config_diff(existing_config, rag_config)
    result["diff"] = diff
    print(f"  Lines added:   {len(diff['added'])}")
    print(f"  Lines removed: {len(diff['removed'])}")
    if diff["added"]:
        print("  New commands:")
        for line in diff["added"][:5]:
            print(f"    + {line}")
    if diff["removed"]:
        print("  Removed commands:")
        for line in diff["removed"][:3]:
            print(f"    - {line}")

    # ── Final output ──────────────────────────────────────────────────────────
    result["status"]      = "READY"
    result["final_config"] = rag_config
    print("\n" + "=" * 70)
    print("✅ DEPLOYMENT-READY CONFIGURATION")
    print("=" * 70)
    print(rag_config)
    print("=" * 70)
    print(f"\nPipeline summary:")
    print(f"  Template base : {result.get('template_lines', 0)} lines")
    print(f"  LLM synthesis : {result.get('llm_lines', 0)} lines")
    print(f"  RAG grounded  : Yes")
    print(f"  Validated     : {confidence}% confidence")
    print(f"  Status        : {result['status']}")
    return result


pipeline_result = full_config_pipeline(
    site_intent="Configure OSPF area 10 with MD5 authentication (key-id 1, key 'Br@nch99') "
                "and BGP peering with upstream ISP at 203.0.113.1 AS 65000. "
                "Add QoS policy BRANCH-STANDARD on WAN interface GigabitEthernet0/0.",
    platform="Cisco IOS-XE",
    existing_config="router ospf 1\n area 10\n!"
)


---
## Summary — Templates + LLM = Best of Both Worlds

### When to Use Each Approach

| Scenario | Use Template | Use LLM | Use RAG+LLM |
|----------|-------------|---------|-------------|
| Standard branch rollout | ✅ | | |
| Novel feature combination | | ✅ | |
| Cross-vendor from same intent | | ✅ | |
| High-risk, audit-required | | | ✅ |
| Platform not in training data | | | ✅ |

### The Four-Stage Pipeline

```
1. Template  →  Deterministic base for predictable 80%
     │
2. LLM Synth →  Handle novel requirements templates can't cover
     │
3. RAG       →  Ground generation in retrieved vendor docs
     │           (prevents knowledge boundary hallucinations)
4. Critic    →  Validate every command before deployment
```

### Key Insights

1. **Hallucination ≠ error message.** The LLM generates deprecated commands
   *confidently*. Confidence is not a reliability signal — validation is.

2. **RAG prevents hallucination by construction.** If the retrieved docs don't
   contain a command, the model can't use it.

3. **Templates eliminate drift.** The same Jinja2/string.Template generates
   byte-for-byte identical configs for every site — no mental model variation.

4. **The Critic is separate from the Generator.** Never ask the generator to
   validate its own output — use an independent model call.

### What's Next

- **Chapter 23**: Automated Incident Response — closing the loop from
  detection to diagnosis to configuration fix, fully automated

> Configuration drift starts on day one. The only way to stop it is to
> eliminate the manual translation step entirely — templates for structure,
> LLM for intelligence, RAG for accuracy.
